# ExDark training pipeline — Colab
Upload and extract the project ZIP under /content, select a GPU runtime, and place the author-downloaded ExDark files in Drive. No dataset or weights are bundled. Execute stages in order; choose paths below. Colab runtime package changes can require a restart.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys
# Auto-detect lowlight project directory in Colab
target_dir = None
for root, dirs, files in os.walk("/content"):
    if "lowlight" in dirs and os.path.isfile(os.path.join(root, "lowlight", "engine.py")):
        target_dir = root
        break

if target_dir:
    os.chdir(target_dir)
    if target_dir not in sys.path:
        sys.path.insert(0, target_dir)
    print(f"Project directory set to: {target_dir}")
else:
    print("Warning: 'lowlight' folder not found under /content yet. Please upload/unzip project files.")

!nvidia-smi

In [ ]:
# Install required dependencies for Colab environment
%pip install ultralytics opencv-python Pillow PyYAML pytest

Restart the session after installation if Colab requests it, then rerun the mount and directory cell. The following directory is persistent; do not store long training runs only in /content.

In [ ]:
import os, sys
from pathlib import Path
if 'target_dir' in globals() and target_dir and os.path.exists(target_dir):
    os.chdir(target_dir)
    if target_dir not in sys.path: sys.path.insert(0, target_dir)

BASE = Path("/content/drive/MyDrive/LowLightResearch")
BASE.mkdir(parents=True, exist_ok=True)
RAW = BASE / "raw"
DATA = BASE / "exdark"
RUNS = BASE / "runs"
!python -m pytest -q

In [ ]:
!python -m lowlight prepare --images "{RAW}/ExDark" --annotations "{RAW}/Groundtruth" --metadata "{RAW}/imageclasslist.txt" --origin 1 --out "{DATA}"

In [ ]:
!python inspect_labels.py --data "{DATA}" --out "{BASE}/inspection" --count 24

Open the inspection images and verify boxes before continuing. New output folders are required for new runs. Reduce batch size if GPU memory is insufficient.

In [ ]:
!python -m lowlight baseline --data "{DATA}" --out "{RUNS}/baseline" --epochs 80 --batch 4 --device cuda:0

In [ ]:
!python -m lowlight pretrain --data "{DATA}" --detector "{RUNS}/baseline/weights/best.pt" --out "{RUNS}/enhancer" --epochs 50 --batch 4 --augment --amp --device cuda:0

In [ ]:
!python -m lowlight guided --data "{DATA}" --detector "{RUNS}/baseline/weights/best.pt" --init "{RUNS}/enhancer/best.pt" --out "{RUNS}/guided" --epochs 30 --batch 4 --lr 0.00005 --det-weight 0.1 --augment --amp --device cuda:0

## End-to-End Joint Fine-tuning (JE-YOLO)
Train both enhancer and detector simultaneously end-to-end (BatchNorm running stats are frozen by default for small-batch stability):

In [ ]:
!python -m lowlight joint --data "{DATA}" --detector "{RUNS}/baseline/weights/best.pt" --init "{RUNS}/enhancer/best.pt" --out "{RUNS}/joint" --epochs 30 --batch 4 --lr 0.00005 --det-lr 0.00001 --det-weight 0.1 --augment --amp --device cuda:0

Only after fixing hyperparameters on validation, run the final test evaluation. Export each stage using its own enhancer checkpoint so guided and joint detectors are evaluated on their matched enhanced images.

In [ ]:
!python -m lowlight export --data "{DATA}" --out "{BASE}/enhanced_guided" --enhancer "{RUNS}/guided/best.pt" --splits val test --device cuda:0
!python -m lowlight export --data "{DATA}" --out "{BASE}/enhanced_joint" --enhancer "{RUNS}/joint/best.pt" --splits val test --device cuda:0
!python -m lowlight evaluate --data "{DATA}" --detector "{RUNS}/baseline/weights/best.pt" --split test --batch 8 --out "{BASE}/results/raw" --device cuda:0
!python -m lowlight evaluate --data "{BASE}/enhanced_guided" --detector "{RUNS}/baseline/weights/best.pt" --split test --batch 8 --out "{BASE}/results/guided" --device cuda:0
!python -m lowlight evaluate --data "{BASE}/enhanced_joint" --detector "{RUNS}/joint/detector_best.pt" --split test --batch 8 --out "{BASE}/results/joint" --device cuda:0
!python compare_results.py --results "{BASE}/results/raw/metrics.json" "{BASE}/results/guided/metrics.json" "{BASE}/results/joint/metrics.json" --out "{BASE}/results/comparison.csv" --per-class-out "{BASE}/results/per_class.csv"

In [ ]:
# Benchmark combined parameter count and warmed end-to-end GPU latency
!python -m lowlight benchmark --enhancer "{RUNS}/joint/best.pt" --detector "{RUNS}/joint/detector_best.pt" --size 640 --runs 50 --warmup 10 --device cuda:0 --out "{BASE}/results/benchmark.json"